In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from model_ranking import (
    to_target_transfer_correlations,
    correlation_table,
    load_transfer_metric_results,
    avg_correlation_table,
)

INFO: P [MainThread] 2026-07-23 12:36:54,215 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nuclei_performance_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/transfer_performance_MAP_scores.json"
with open(nuclei_performance_path, "r") as f:
    nuclei_performance_scores = json.load(f)["performance_scores"]

nuclei_performance_scores.pop("DSB2018", None)

{'BC_IN_model2': 0.4842800498008728,
 'HN_IN_model2': 0.5206860899925232,
 'Hst_IN_model3': 0.31693270802497864,
 '895_IN_model2': 0.43998968601226807,
 '1410_IN_model1': 0.49643415212631226}

# Gauss ARE

### Nuclei

In [6]:
base_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/ARE"
#selected_augmentations = ["a0-005", "a005-01", "a01-02"]
selected_augmentations = ["a01-02"]

for aug in selected_augmentations:
    file_name = f"transfer_Gauss_{aug}_Forg_ARE_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    consistency_scores.pop("DSB2018", None)
    targets = list(consistency_scores.keys())
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=nuclei_performance_scores,
        invert_transfer_metric=True,
        )
    df_nuclei_gauss_ARE = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, task='Nuclei')
    print(df_nuclei_gauss_ARE)
    df_nuclei_avg_gauss_ARE = avg_correlation_table(df_nuclei_gauss_ARE)
    print(df_nuclei_avg_gauss_ARE)
            
        

Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/ARE/transfer_Gauss_a01-02_Forg_ARE_scores.json
                    kt  kt pval  s rho  s rho pval    pr  pr pval
Task   targets                                                   
Nuclei BBBC039    0.40     0.44    0.6        0.34  0.50     0.39
       Hoechst    1.00     0.01    1.0        0.02  0.93     0.02
       S_BIAD895  0.67     0.33    0.8        0.33  0.94     0.06
       S_BIAD634  0.80     0.07    0.9        0.13  0.79     0.11
        KT_avg    KT_std  SR_avg    SR_std  PR_avg    PR_std
Task                                                        
Nuclei  0.7175  0.251446   0.825  0.170783    0.79  0.205102


# DO ARE

### Nuclei

In [7]:
base_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/feature_perturbation_consistency/DO/ARE"
#selected_augmentations = ["a005", "a01", "a02", "a03", "a04", "a05"]
selected_augmentations = ["a02"]

for aug in selected_augmentations:
    file_name = f"transfer_DO_{aug}_Forg_ARE_scores.json"
    consistency_path = Path(base_path) / file_name
    results = load_transfer_metric_results(consistency_path)
    consistency_scores = results["transfer_scores"]
    consistency_scores.pop("DSB2018", None)
    targets = list(consistency_scores.keys())
    KT_scores, SP_scores, PE_scores = to_target_transfer_correlations(
        targets=targets,
        transfer_metric_per_target=consistency_scores,
        performance_per_target=nuclei_performance_scores,
        invert_transfer_metric=True,
        )

    df_nuclei_DO_ARE = correlation_table(KT_scores, SP_scores, PE_scores, targets=targets, task='Nuclei')
    print(df_nuclei_DO_ARE)
    df_nuclei_avg_DO_ARE = avg_correlation_table(df_nuclei_DO_ARE)
    print(df_nuclei_avg_DO_ARE)


Loaded Transfer metric results from: /g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/feature_perturbation_consistency/DO/ARE/transfer_DO_a02_Forg_ARE_scores.json
                   kt  kt pval  s rho  s rho pval    pr  pr pval
Task   targets                                                  
Nuclei BBBC039    0.4     0.50    0.5        0.48  0.95     0.02
       Hoechst    1.0     0.02    1.0        0.02  0.93     0.02
       S_BIAD895  1.0     0.08    1.0        0.08  0.72     0.28
       S_BIAD634  0.8     0.09    0.9        0.09  0.78     0.12
        KT_avg    KT_std  SR_avg    SR_std  PR_avg    PR_std
Task                                                        
Nuclei     0.8  0.282843    0.85  0.238048   0.845  0.112694


# SEG

In [8]:
targets = ["BBBC039", "Hoechst", "S_BIAD895", "S_BIAD634"]
radii = {
    "BBBC039": 16,
    "Hoechst": 22,
    "S_BIAD895": 12,
    "S_BIAD634": 90,
}
agree_ratio = 0.75
perf_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/transfer_performance_MAP_scores.json"
base_SEG_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/ensemble_scores"
SEG_filenames = {
    "BBBC039": "SEG_results1.pkl",
    "Hoechst": "SEG_results.pkl",
    "S_BIAD895": "SEG_results.pkl",
    "S_BIAD634": "SEG_results1.pkl",
}

In [9]:
performance_path = "/g/kreshuk/talks/consistency_results/Instance_segmentation/nuclei/transfer_results/consistency/transfer_performance_MAP_scores.json"
with open(performance_path, "r") as f:
    performance_scores = json.load(f)["performance_scores"]

In [10]:
import pickle
from typing import Dict


per_target_SEG_uneq: Dict[str, Dict[str, float]] = {}
per_target_SEG_eq: Dict[str, Dict[str, float]] = {}
for target in targets:
    r = radii[target]
    print(f"Radius: {r}, Agree ratio: {agree_ratio}")

    SEG_path = Path(base_SEG_path) / f"to_{target}" / SEG_filenames[target]
    with open(SEG_path, "rb") as f:
        SEG_results = pickle.load(f)
    
    methods = SEG_results["methods"]
    eq_f1s = SEG_results["eq_f1s"]
    uneq_f1s = SEG_results["uneq_f1s"]

    f1s_uneq: Dict[str, float] = {}
    f1s_eq: Dict[str, float] = {}
    for i, m in enumerate(methods):
        f1s_uneq[m] = uneq_f1s[r][agree_ratio][i]
        f1s_eq[m] = eq_f1s[r][agree_ratio][i]
    per_target_SEG_uneq[target] = f1s_uneq
    per_target_SEG_eq[target] = f1s_eq


KT_scores_eq, SP_scores_eq, PE_scores_eq = to_target_transfer_correlations(
    targets=targets,
    transfer_metric_per_target=per_target_SEG_eq,
    performance_per_target=performance_scores,
    invert_transfer_metric=False,
    )

correlation_df_eq = correlation_table(
    KT_scores_eq, SP_scores_eq, PE_scores_eq, targets=targets, task='Nuclei'
)
print("eq_weight")
print(correlation_df_eq)
df_avg_eq = avg_correlation_table(correlation_df_eq)
print(df_avg_eq)

KT_scores_uneq, SP_scores_uneq, PE_scores_uneq = to_target_transfer_correlations(
    targets=targets,
    transfer_metric_per_target=per_target_SEG_uneq,
    performance_per_target=performance_scores,
    invert_transfer_metric=False,
)
correlation_df_uneq = correlation_table(
    KT_scores_uneq, SP_scores_uneq, PE_scores_uneq, targets=targets, task='Nuclei'
)
print("uneq_weight")
print(correlation_df_uneq)
df_avg_uneq = avg_correlation_table(correlation_df_uneq)
print(df_avg_uneq)

Radius: 16, Agree ratio: 0.75
Radius: 22, Agree ratio: 0.75
Radius: 12, Agree ratio: 0.75
Radius: 90, Agree ratio: 0.75
eq_weight
                   kt  kt pval  s rho  s rho pval    pr  pr pval
Task   targets                                                  
Nuclei BBBC039    0.0     1.00   -0.1        0.92 -0.66     0.23
       Hoechst    0.6     0.28    0.8        0.13  0.79     0.11
       S_BIAD895  0.0     1.00   -0.2        0.92 -0.45     0.55
       S_BIAD634 -0.6     0.26   -0.8        0.14 -0.67     0.22
        KT_avg    KT_std  SR_avg    SR_std  PR_avg    PR_std
Task                                                        
Nuclei     0.0  0.489898  -0.075  0.660177 -0.2475  0.699065
uneq_weight
                    kt  kt pval  s rho  s rho pval    pr  pr pval
Task   targets                                                   
Nuclei BBBC039    0.00     1.00   -0.1        0.96 -0.57     0.32
       Hoechst    0.60     0.24    0.8        0.12  0.82     0.09
       S_BIAD895  0.3

# Combined CSV results

In [11]:
avg_dataframes = [
    df_nuclei_avg_gauss_ARE,
    df_nuclei_avg_DO_ARE,
    df_avg_eq,
    df_avg_uneq
]

dataframes = [
    df_nuclei_gauss_ARE,
    df_nuclei_DO_ARE,
    correlation_df_eq,
    correlation_df_uneq
]
metric_names = [
    "Gauss_ARS",
    "DO_ARS",
    "SEG_eq",
    "SEG_uneq"
]

In [12]:
import numpy as np
import pandas as pd

flat_dataframes = dataframes
flat_avg_dataframes = avg_dataframes

if len(flat_dataframes) != len(metric_names):
    raise ValueError(
        f"Length mismatch: {len(flat_dataframes)} dataframes vs {len(metric_names)} metric names"
    )

if len(flat_avg_dataframes) != len(metric_names):
    raise ValueError(
        f"Length mismatch: {len(flat_avg_dataframes)} avg dataframes vs {len(metric_names)} metric names"
    )

# Build target columns from the actual per-target tables to avoid empty columns.
all_targets = []
for df_metric in flat_dataframes:
    df_reset = df_metric.reset_index()
    if "targets" not in df_reset.columns:
        raise ValueError("Missing 'targets' column in one of the correlation tables")
    all_targets.extend(df_reset["targets"].dropna().astype(str).tolist())

# Preserve target appearance order while removing duplicates.
target_columns = list(dict.fromkeys(all_targets))

# (corr_name_in_df, pval_name_in_df, avg_col_in_avg_df, std_col_in_avg_df)
corr_specs = [
    ("kt", "kt pval", "KT_avg", "KT_std"),
    ("s rho", "s rho pval", "SR_avg", "SR_std"),
    ("pr", "pr pval", "PR_avg", "PR_std"),
]

combined_rows = []

for metric_name, df_metric, df_metric_avg in zip(metric_names, flat_dataframes, flat_avg_dataframes):
    df_reset = df_metric.reset_index()

    if "targets" not in df_reset.columns:
        raise ValueError(f"Missing 'targets' column for metric '{metric_name}'")

    for corr_type, pval_col, avg_col, std_col in corr_specs:
        # Score row: includes per-target correlation values + aggregate mean/std.
        corr_row = {
            "metric_name": metric_name,
            "corr_type": corr_type,
            "pval_type": "score",
            "corr_avg": df_metric_avg[avg_col].iloc[0],
            "corr_std": df_metric_avg[std_col].iloc[0],
        }

        # P-value row: includes per-target p-values (aggregate stats are not defined here).
        pval_row = {
            "metric_name": metric_name,
            "corr_type": corr_type,
            "pval_type": "pval",
            "corr_avg": np.nan,
            "corr_std": np.nan,
        }

        for target in target_columns:
            target_match = df_reset[df_reset["targets"].astype(str) == str(target)]
            if target_match.empty:
                corr_row[target] = np.nan
                pval_row[target] = np.nan
            else:
                corr_row[target] = target_match[corr_type].iloc[0]
                pval_row[target] = target_match[pval_col].iloc[0]

        combined_rows.append(corr_row)
        combined_rows.append(pval_row)

combined_results_df = pd.DataFrame(combined_rows)

# Match ordering used in the provided CSV example.
sort_orders = {
    "kt": 0,
    "s rho": 1,
    "pr": 2,
    "score": 0,
    "pval": 1,
}

combined_results_df = combined_results_df.sort_values(
    by=["metric_name", "corr_type", "pval_type"],
    key=lambda s: s.map(sort_orders).fillna(s),
).reset_index(drop=True)

# Keep a consistent column order for export.
combined_results_df = combined_results_df[
    ["metric_name", "corr_type", "pval_type", "corr_avg", "corr_std", *target_columns]
]

combined_results_df

,metric_name,corr_type,pval_type,corr_avg,corr_std,BBBC039,Hoechst,S_BIAD895,S_BIAD634
0,DO_ARS,kt,score,0.8000,0.282843,0.40,1.00,1.00,0.80
1,DO_ARS,kt,pval,NaN,NaN,0.50,0.02,0.08,0.09
2,DO_ARS,s rho,score,0.8500,0.238048,0.50,1.00,1.00,0.90
3,DO_ARS,s rho,pval,NaN,NaN,0.48,0.02,0.08,0.09
4,DO_ARS,pr,score,0.8450,0.112694,0.95,0.93,0.72,0.78
5,DO_ARS,pr,pval,NaN,NaN,0.02,0.02,0.28,0.12
6,Gauss_ARS,kt,score,0.7175,0.251446,0.40,1.00,0.67,0.80
7,Gauss_ARS,kt,pval,NaN,NaN,0.44,0.01,0.33,0.07
8,Gauss_ARS,s rho,score,0.8250,0.170783,0.60,1.00,0.80,0.90
9,Gauss_ARS,s rho,pval,NaN,NaN,0.34,0.02,0.33,0.13


In [13]:
from pathlib import Path

output_csv_path = Path("/g/kreshuk/talks/model_ranking/notebooks/thesis/nuclei/instance/nuclei_instance_correlations.csv")
combined_results_df.to_csv(output_csv_path, index=False)
print(f"Saved combined results to: {output_csv_path}")

Saved combined results to: /g/kreshuk/talks/model_ranking/notebooks/thesis/nuclei/instance/nuclei_instance_correlations.csv
